# 🔴 미션 3 (선택) — 어제의 덧셈형 채점기와 오늘의 내적을 비교한다

논문 §3.2.1 은 내적을 쓰는 이유를 이렇게 적었다.
*"dot-product attention is **much faster and more space-efficient** in practice"*

정말 그런지 직접 잰다.

⚠️ **품질을 비교하는 것이 아니다.** 두 채점기 모두 학습하지 않았으므로
"어느 쪽이 더 똑똑한가"는 잴 수 없다. 오늘 재는 것은 **비용**이다.

**제출**
1. 비교표 (시간 · 메모리 · 파라미터)
2. 한 줄: 논문의 주장이 맞았는가

In [ ]:
import math
import sys
import time

# 노트북(cwd=이 폴더)에서도, .py 를 상위에서 돌려도 attn/koplot 을 찾게 한다.
# ⚠️ __file__ 은 주피터 커널에 정의되지 않는다 — 노트북에서 NameError 로 죽는다.
sys.path[:0] = ["..", "."]

import torch
import torch.nn as nn

torch.manual_seed(0)

## 1. 두 채점기를 나란히 놓는다

In [ ]:
def 점수_내적(Q, K):
    """오늘의 것 — 행렬곱 한 번."""
    return Q @ K.T / math.sqrt(Q.size(-1))


def 점수_덧셈형(Q, K, Wq, Wk, v):
    """어제의 것 — 짝마다 작은 신경망을 통과한다.

    모든 짝을 한꺼번에 하려면 (L, L, d) 짜리 중간 텐서를 만들어야 한다.
    이 한 줄이 오늘 비교의 핵심이다.
    """
    h = torch.tanh(Wq(Q).unsqueeze(1) + Wk(K).unsqueeze(0))   # (L, L, d)
    return h @ v

## 2. 시간을 잰다

처음 한두 번은 준비 과정이 섞여 느리게 나온다. 그래서 **워밍업**을 먼저 돌린다.

In [ ]:
def 재기(L, d, 반복=20):
    Q, K = torch.randn(L, d), torch.randn(L, d)
    Wq, Wk = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
    v = torch.randn(d)

    for _ in range(3):                      # 워밍업
        점수_내적(Q, K)
        점수_덧셈형(Q, K, Wq, Wk, v)

    t0 = time.perf_counter()
    for _ in range(반복):
        점수_내적(Q, K)
    내적시간 = (time.perf_counter() - t0) / 반복

    t0 = time.perf_counter()
    for _ in range(반복):
        점수_덧셈형(Q, K, Wq, Wk, v)
    덧셈시간 = (time.perf_counter() - t0) / 반복

    return 내적시간, 덧셈시간


print(f"{'문장 길이':>8} | {'내적 (ms)':>10} | {'덧셈형 (ms)':>12} | {'몇 배 느린가':>12}")
print("-" * 52)
for L in (64, 128, 256, 512):
    a, b = 재기(L, d=64)
    print(f"{L:>8} | {a * 1000:>10.3f} | {b * 1000:>12.3f} | {b / a:>11.1f}배")

   문장 길이 |    내적 (ms) |     덧셈형 (ms) |      몇 배 느린가
----------------------------------------------------
      64 |      0.029 |        0.264 |         9.0배
     128 |      0.058 |        0.812 |        13.9배
     256 |      0.101 |        4.912 |        48.8배
     512 |      0.133 |       19.044 |       143.2배


## 3. 메모리를 견준다

시간은 기계 사정을 타지만, **중간 텐서의 크기는 구조적으로 정해진다.**

In [ ]:
d = 64
print(f"{'문장 길이':>8} | {'내적 (L×L)':>14} | {'덧셈형 (L×L×d)':>16} | {'몇 배':>7}")
print("-" * 56)
for L in (64, 128, 256, 512):
    a = L * L * 4 / 1024 ** 2
    b = L * L * d * 4 / 1024 ** 2
    print(f"{L:>8} | {a:>11.2f} MB | {b:>13.2f} MB | {b / a:>6.0f}배")

print(f"\n비율은 언제나 정확히 d = {d} 배다. 덧셈형은 짝마다 길이 {d} 짜리 벡터를 만들어야 하기 때문.")

   문장 길이 |       내적 (L×L) |      덧셈형 (L×L×d) |     몇 배
--------------------------------------------------------
      64 |        0.02 MB |          1.00 MB |     64배
     128 |        0.06 MB |          4.00 MB |     64배
     256 |        0.25 MB |         16.00 MB |     64배
     512 |        1.00 MB |         64.00 MB |     64배

비율은 언제나 정확히 d = 64 배다. 덧셈형은 짝마다 길이 64 짜리 벡터를 만들어야 하기 때문.


## 4. 파라미터를 센다

In [ ]:
d = 64
print(f"내적형  : {0:>6,} 개   ← 점수를 내는 데 학습 파라미터가 필요 없다")
print(f"덧셈형  : {2 * d * d + d:>6,} 개   ← Wq({d}×{d}) + Wk({d}×{d}) + v({d})")

내적형  :      0 개   ← 점수를 내는 데 학습 파라미터가 필요 없다
덧셈형  :  8,256 개   ← Wq(64×64) + Wk(64×64) + v(64)


## 5. 한 줄로 정리한다

In [ ]:
# TODO: 내 말로 한 줄
print("""
논문의 주장("더 빠르고 메모리를 덜 쓴다")은 ______________.
내가 잰 것은 어느 쪽이 더 똑똑한가가 아니라, 어느 쪽이 더 ______다.
""")


논문의 주장("더 빠르고 메모리를 덜 쓴다")은 ______________.
내가 잰 것은 어느 쪽이 더 똑똑한가가 아니라, 어느 쪽이 더 ______다.

